# Mathematical Foundations for Machine Learning
## Linear Algebra · Probability Theory · Optimization · Information Theory

**TL;DR:** The four mathematical pillars of ML — if you cannot derive these from scratch, you cannot read ML papers. Stanford CS229 opens with all four. This notebook gives you the minimum mathematical literacy required to understand LoRA, Pairformer, EGNN, and every other model in this curriculum.

## Beginner Teaching Frame

**Who should fully work through this notebook:** students with little or no ML background.

**How to study it on a first pass:**
- read every markdown section before touching the code
- run the code in small chunks
- paraphrase each concept in your own words
- write one tiny example from memory after finishing

**Common traps:** memorizing syntax without understanding the data flow, skipping probability, and rushing past train/test split discipline.


## Canonical Resource Upgrade

If you need a second explanation, use these first:
- [CS50P](https://cs50.harvard.edu/python/) for programming fundamentals
- [Harvard Stat 110](https://stat110.hsites.harvard.edu/) for probability intuition
- [scikit-learn MOOC](https://inria.github.io/scikit-learn-mooc/) for correct ML workflow
- [PyTorch Tutorials](https://docs.pytorch.org/tutorials/) once you reach the deep learning notebooks


## 📄 Primary Literature — Key Papers

These results are based on peer-reviewed publications. Read the originals to go deeper:

- **Bishop 2006** — *Pattern Recognition and Machine Learning (PRML) — free PDF*  
  [https://www.microsoft.com/en-us/research/uploads/prod/2006/01/Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf](https://www.microsoft.com/en-us/research/uploads/prod/2006/01/Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf)
- **Goodfellow et al. 2016** — *Deep Learning textbook (free online)*  
  [https://www.deeplearningbook.org/](https://www.deeplearningbook.org/)


## Prerequisites & Learning Path

| | |
|--|--|
| Prerequisites | High-school calculus and linear algebra (some exposure) |
| You Are Here | Module 00/08 — Mathematical Foundations (the math behind all ML) |
| Next Steps | 00/06 PyTorch -> 05/01 Deep Learning -> 07/01 AlphaFold3 |
| Goal | Derive PCA, MLE, gradient descent, and cross-entropy from scratch |

### From Scratch? Start Here:
1. [3Blue1Brown — Essence of Linear Algebra](https://www.youtube.com/playlist?list=PLZHQObOWTQDPD3MizzM2xVFitgF8hE_ab) — 15 videos, 3 hours
2. [3Blue1Brown — Essence of Calculus](https://www.youtube.com/playlist?list=PLZHQObOWTQDMsr9K-rj53DwVRMYO3t5Yr) — 12 videos, 2.5 hours
3. [Khan Academy Probability](https://www.khanacademy.org/math/statistics-probability) — free, at your own pace
4. This notebook — brings it all together for ML

**Time:** 12-20 hours | **Difficulty:** 4/5

### Cross-References
- Builds on: High-school math (vectors, matrices, calculus basics)
- Used by: EVERY other notebook in this curriculum — this is the foundation
- Most critical for: 07/01 (FAPE loss derivation), 06/02 (graph Laplacian), 10/01 (LoRA rank theory)

---
## PART 1 — Linear Algebra

Linear algebra is the language of ML. Every neural network forward pass is a sequence of matrix multiplications. PCA, SVD, and eigendecomposition appear in dimensionality reduction, LoRA, graph neural networks, and attention mechanisms.

**Key ideas:**
- Vectors represent data points, weights, or directions in space
- Matrix multiplication = composition of linear transformations
- Eigendecomposition reveals the intrinsic structure of a symmetric matrix
- SVD generalises eigendecomposition to rectangular matrices and is the backbone of PCA and LoRA

**Why it matters here:**

| Concept | Where it appears |
|---|---|
| Dot product / cosine similarity | Attention scores `QK^T` |
| Matrix multiply | Every linear layer |
| Eigendecomposition | Spectral graph theory (GNNs) |
| SVD | PCA, LoRA low-rank update |
| Rank | LoRA rank `r` — controls parameter count |

In [ ]:
# Linear Algebra: SVD and PCA
import numpy as np

# SVD: A = U Σ V^T
# Every matrix decomposes into rotation × scaling × rotation
np.random.seed(42)
A = np.array([[3, 1, 1], [1, 2, 0], [0, 1, 4]], dtype=float)
U, s, Vt = np.linalg.svd(A)

print("SVD of A:")
print(f"  A = U Σ V^T")
print(f"  U shape: {U.shape} (left singular vectors — column space)")
print(f"  Σ (singular values): {s.round(2)}")
print(f"  V^T shape: {Vt.shape} (right singular vectors — row space)")

# Reconstruct A from SVD
A_reconstructed = U @ np.diag(s) @ Vt
print(f"  Reconstruction error: {np.linalg.norm(A - A_reconstructed):.2e}")

# PCA via SVD
X = np.random.randn(50, 5)
X_centered = X - X.mean(0)
U_pca, s_pca, Vt_pca = np.linalg.svd(X_centered, full_matrices=False)
# Principal components = columns of V (rows of V^T)
PC = Vt_pca.T  # (5, 5) — columns are principal components
variance_explained = s_pca**2 / (s_pca**2).sum()
print(f"\nPCA variance explained: {variance_explained.round(3)}")

# Eigendecomposition: A = Q Λ Q^{-1}
eigenvalues, eigenvectors = np.linalg.eig(A)
print(f"\nEigenvalues of A: {eigenvalues.real.round(2)}")

> **Expected output:** SVD reconstruction error: ~1e-15 (near zero), PCA variance explained array, and eigenvalues of A  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

---
## PART 2 — Probability Theory

Probability theory underpins every ML model that makes predictions. Cross-entropy loss, regularization, variational autoencoders, Bayesian neural networks — all of these are statements about probability distributions.

**Key ideas:**
- Bayes theorem: how to update beliefs given data
- MLE: choose parameters that maximise the probability of the observed data
- MAP: MLE with a prior — this IS regularization
- KL divergence: how to measure the distance between two distributions

**Why it matters here:**

| Concept | Where it appears |
|---|---|
| Gaussian likelihood | MSE loss derivation |
| Bayes theorem | Posterior inference, dropout as Bayes |
| MLE | Training any neural network |
| MAP / L2 reg | Ridge regression, weight decay |
| KL divergence | VAE loss, ELBO, distillation |

In [ ]:
# Probability Theory: MLE and KL Divergence
import numpy as np
from scipy import stats

# Maximum Likelihood Estimation (MLE)
# For Gaussian: MLE parameters are sample mean and variance
np.random.seed(42)
true_mu, true_sigma = 2.0, 0.8
data = np.random.normal(true_mu, true_sigma, 200)

mle_mu = data.mean()
mle_sigma = data.std(ddof=0)  # MLE uses N, not N-1
print(f"MLE for Gaussian:")
print(f"  True: μ={true_mu}, σ={true_sigma}")
print(f"  MLE:  μ={mle_mu:.3f}, σ={mle_sigma:.3f}")
print(f"  Log-likelihood: {np.sum(stats.norm.logpdf(data, mle_mu, mle_sigma)):.2f}")

# KL Divergence: D_KL(P||Q) = Σ P(x) log(P(x)/Q(x))
# For two Gaussians: D_KL(N(μ1,σ1) || N(μ2,σ2)) has closed form
def kl_gaussian(mu1, sigma1, mu2, sigma2):
    return (np.log(sigma2/sigma1) +
            (sigma1**2 + (mu1-mu2)**2) / (2*sigma2**2) - 0.5)

print(f"\nKL Divergence:")
print(f"  D_KL(N(2,0.8)||N(2,0.8)) = {kl_gaussian(2,0.8,2,0.8):.4f} (same dist)")
print(f"  D_KL(N(2,0.8)||N(3,0.8)) = {kl_gaussian(2,0.8,3,0.8):.4f}")
print(f"  D_KL(N(2,0.8)||N(2,2.0)) = {kl_gaussian(2,0.8,2,2.0):.4f}")
print(f"  Note: D_KL is NOT symmetric!")
print(f"  D_KL(P||Q) ≥ 0, equals 0 only when P=Q (Gibbs inequality)")

---
## PART 3 — Optimization

Machine learning training is optimization. Understanding gradient descent, adaptive optimizers, convexity, and constrained optimization is essential for diagnosing training failures and understanding model behaviour.

**Key ideas:**
- Gradient descent: follow the steepest descent direction
- Adam: adaptive per-parameter learning rates using running moments of gradients
- Convexity: guarantees a global minimum; neural networks are NOT convex
- Lagrangians: optimization under constraints (SVM, attention softmax)

**Why it matters here:**

| Concept | Where it appears |
|---|---|
| SGD / Adam | Training any model in PyTorch |
| Learning rate schedule | Transformer training (warmup + cosine decay) |
| Convexity | Why logistic regression converges but NN does not |
| Lagrangian | SVM max-margin, attention weight normalisation |
| Saddle points | Why deep networks can still train despite non-convexity |

In [ ]:
# Mathematical Foundations: Summary
import numpy as np
from scipy import stats

np.random.seed(42)

# 1. Linear algebra: matrix decomposition
A = np.random.randn(4, 4)
U, s, Vt = np.linalg.svd(A)
print(f"Condition number of A: {s[0]/s[-1]:.2f}")
print(f"Rank of A: {np.linalg.matrix_rank(A)}")

# 2. Probability: sample from distribution
samples = np.random.normal(0, 1, 1000)
print(f"\nSample stats: mean={samples.mean():.3f}, std={samples.std():.3f}")
print(f"KS test vs N(0,1): p={stats.kstest(samples, 'norm').pvalue:.3f}")

# 3. Optimization: gradient step
def rosenbrock(x): return (1-x[0])**2 + 100*(x[1]-x[0]**2)**2
def rosenbrock_grad(x):
    return np.array([
        -2*(1-x[0]) - 400*x[0]*(x[1]-x[0]**2),
        200*(x[1]-x[0]**2)
    ])

x = np.array([-1.0, 0.5])
for _ in range(1000):
    x -= 1e-3 * rosenbrock_grad(x)
print(f"\nRosenbrock minimum found: {x} (true: [1,1])")

---
## PART 4 — Information Theory

Shannon's information theory provides the theoretical foundation for understanding what neural networks are learning and why particular loss functions work.

**Key ideas:**
- Entropy: the average uncertainty in a distribution
- Cross-entropy: the loss function for classification — measures how well the model distribution matches data
- Mutual information: how much two variables share — critical for understanding coevolution in protein sequences
- KL divergence (revisited): the geometry of probability space

**Why it matters here:**

| Concept | Where it appears |
|---|---|
| Cross-entropy loss | Training every classifier |
| Entropy | Decision tree splits (information gain) |
| Mutual information | MSA coevolution (DCA, AF2 input) |
| KL divergence | VAE ELBO, knowledge distillation |
| Information bottleneck | Why deep nets generalise |

In [ ]:
# Information Theory: Entropy and Mutual Information
import numpy as np
from scipy.stats import entropy as scipy_entropy

# Shannon entropy: H(X) = -Σ p(x) log2 p(x)
def shannon_entropy(probs):
    probs = np.array(probs)
    probs = probs[probs > 0]  # log(0) undefined
    return -np.sum(probs * np.log2(probs))

# Nucleotide frequency examples
uniform_dna = [0.25, 0.25, 0.25, 0.25]  # max entropy
gc_rich = [0.1, 0.4, 0.4, 0.1]           # GC-rich sequence
coding_region = [0.3, 0.2, 0.2, 0.3]     # coding region

print("Shannon Entropy of nucleotide distributions:")
for name, dist in [("Uniform", uniform_dna), ("GC-rich", gc_rich), ("Coding", coding_region)]:
    H = shannon_entropy(dist)
    print(f"  {name}: H = {H:.3f} bits (max = {np.log2(4):.3f})")

# Cross-entropy: H(P, Q) = -Σ P(x) log Q(x) — used as loss in ML
P = np.array([0.8, 0.1, 0.1])  # true distribution
Q = np.array([0.7, 0.2, 0.1])  # predicted
cross_ent = -np.sum(P * np.log(Q))
kl = np.sum(P * np.log(P/Q))
print(f"\nCross-entropy H(P,Q) = {cross_ent:.4f}")
print(f"KL D_KL(P||Q) = {kl:.4f}")
print(f"Relationship: H(P,Q) = H(P) + D_KL(P||Q) = {shannon_entropy(P)/np.log2(np.e) + kl:.4f}")

---
## PART 5 — Calculus for Backpropagation

Backpropagation is the engine of deep learning. It is nothing more than the chain rule of calculus applied systematically to a computational graph. Understanding this from first principles demystifies PyTorch's `autograd` and allows you to debug gradient flow issues.

**Key ideas:**
- Chain rule: how derivatives compose through function composition
- Computational graph: a directed acyclic graph where nodes are operations and edges carry values
- Forward pass: compute the loss (left to right)
- Backward pass: compute gradients (right to left, accumulating via chain rule)
- Gradient check: numerically verify analytical gradients

**Why it matters here:**

| Concept | Where it appears |
|---|---|
| Chain rule | Every backward() call in PyTorch |
| ReLU gradient | Dead neurons, gradient vanishing |
| Jacobian | Equivariant networks, SE(3) layers |
| Gradient check | Debugging custom loss functions (e.g. FAPE) |
| Hessian | Second-order optimizers, loss landscape analysis |

In [ ]:
# Mathematical Foundations: Summary
import numpy as np
from scipy import stats

np.random.seed(42)

# 1. Linear algebra: matrix decomposition
A = np.random.randn(4, 4)
U, s, Vt = np.linalg.svd(A)
print(f"Condition number of A: {s[0]/s[-1]:.2f}")
print(f"Rank of A: {np.linalg.matrix_rank(A)}")

# 2. Probability: sample from distribution
samples = np.random.normal(0, 1, 1000)
print(f"\nSample stats: mean={samples.mean():.3f}, std={samples.std():.3f}")
print(f"KS test vs N(0,1): p={stats.kstest(samples, 'norm').pvalue:.3f}")

# 3. Optimization: gradient step
def rosenbrock(x): return (1-x[0])**2 + 100*(x[1]-x[0]**2)**2
def rosenbrock_grad(x):
    return np.array([
        -2*(1-x[0]) - 400*x[0]*(x[1]-x[0]**2),
        200*(x[1]-x[0]**2)
    ])

x = np.array([-1.0, 0.5])
for _ in range(1000):
    x -= 1e-3 * rosenbrock_grad(x)
print(f"\nRosenbrock minimum found: {x} (true: [1,1])")

---
## Resources

### 1 — Theory: Foundations & Math
- **Linear Algebra:** Gilbert Strang MIT OCW — eigenvectors are the DNA of linear algebra
- **Probability:** Jaynes "Probability Theory: The Logic of Science" — rigorous Bayesian perspective
- **Optimization:** Boyd & Vandenberghe "Convex Optimization" — free PDF, definitive reference
- **Information Theory:** Cover & Thomas "Elements of Information Theory" — the textbook
- **ML Math:** Mathematics for Machine Learning (Deisenroth et al.) — free PDF, purpose-built

### 2 — Must-Have Popular Resources
#### 🟢 Start Here (Zero Math Background)
- 🆓 **3Blue1Brown Linear Algebra** — [youtube.com/playlist](https://www.youtube.com/playlist?list=PLZHQObOWTQDPD3MizzM2xVFitgF8hE_ab) — 15 videos, best visual explanation of linear algebra ever made
- 🆓 **3Blue1Brown Calculus** — [youtube.com/playlist](https://www.youtube.com/playlist?list=PLZHQObOWTQDMsr9K-rj53DwVRMYO3t5Yr) — 12 videos, understand derivatives and integrals
- 🆓 **Khan Academy Statistics** — [khanacademy.org/math/statistics-probability](https://www.khanacademy.org/math/statistics-probability) — free, self-paced, mastery-based
- 🆓 **Mathematics for Machine Learning (MML)** — [mml-book.github.io](https://mml-book.github.io/) — free PDF, purpose-built for ML, 400 pages
- **Book:** [Mathematics for Machine Learning (MML)](https://mml-book.github.io/) — free, GitHub-hosted, 400 pages, PERFECT for this curriculum
- **Book:** Pattern Recognition & Machine Learning (Bishop) — rigorous, probabilistic perspective, free PDF
- **Course:** [CS229 Stanford Machine Learning (Andrew Ng)](http://cs229.stanford.edu/) — free lecture notes cover all 4 pillars
- **Video:** [3Blue1Brown — Essence of Linear Algebra](https://www.youtube.com/playlist?list=PLZHQObOWTQDPD3MizzM2xVFitgF8hE_ab) — 15 videos, 4M views
- **Video:** [3Blue1Brown — Neural Networks](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi) — backprop visually explained
- **GitHub:** [ageron/handson-ml3](https://github.com/ageron/handson-ml3) — 28k stars, hands-on exercises
- **Kaggle:** [Linear Algebra course](https://www.kaggle.com/learn) — applied math for ML

### 3 — Practicals: Hands-On
- **Exercise 1:** Implement PCA from scratch (no sklearn), verify eigenvalues match SVD result
- **Exercise 2:** Derive gradient of softmax cross-entropy and implement manual backward pass
- **Exercise 3:** Implement Adam optimizer from scratch, compare to PyTorch's built-in
- **Exercise 4:** Show numerically that L2 regularization = MAP with Gaussian prior
- GitHub: [ddbourgin/numpy-ml](https://github.com/ddbourgin/numpy-ml) — ML algorithms in pure NumPy, 15k stars

### 4 — Real-World Problems
- **structural biology research labs / computational biology ML teams:** Every paper they publish requires understanding of all 4 math pillars
- **Application:** Derive the FAPE loss in Module 07 from scratch — it requires SE(3) geometry + optimization
- **Application:** Derive why LoRA works (Module 10) using SVD and intrinsic dimensionality theory
- **Application:** Show that the graph Laplacian eigenvalues encode protein graph connectivity (Module 06)

### 5 — Interview Tips
- **Must know:** Derive PCA from SVD — most common ML theory question
- **Must know:** Why is MLE equivalent to least squares for Gaussian noise? (derive in 3 lines)
- **Must know:** What is KL divergence and why is it not symmetric?
- **Common mistake:** Saying "gradient descent finds global minimum" — only for convex functions
- **Common mistake:** Confusing bias-variance tradeoff with regularization — related but distinct
- **Pro tip:** At structural biology research labs/computational biology ML teams interviews, expect to derive FAPE, attention, or LoRA on a whiteboard

### 6 — Hot Industry Topics
- **Trending:** [Geometric Deep Learning (Bronstein 2021)](https://arxiv.org/abs/2104.13478) — unifying ML through symmetry (group theory)
- **Trending:** Neural Tangent Kernel — infinite-width NN = Gaussian Process, explains generalisation
- **Trending:** [Mechanistic Interpretability](https://github.com/neelnanda-io/TransformerLens) — reverse-engineering what neural networks compute
- **Build:** Implement a complete ML algorithm (linear regression, logistic regression, SVM) using ONLY NumPy from scratch and verify against sklearn
- **Build:** Implement and visualise the Adam optimizer step-by-step, show bias correction effect in first 10 steps

## Interview Q&A — Mathematical Foundations for ML

**Q: Why does PCA use eigenvectors of the covariance matrix?**
A: PCA finds directions of maximum variance. The covariance matrix C = (1/n)X^TX encodes how features vary together. Its eigenvectors are orthogonal directions; eigenvalues tell you variance along each direction. The top-k eigenvectors = top-k principal components = maximum variance directions. SVD is numerically preferred: X = UΣV^T, columns of V are principal components.

**Q: Why is MLE equivalent to minimizing cross-entropy for classification?**
A: MLE: maximize ∏P(y_i|x_i) → take log → maximize Σlog P(y_i|x_i). Cross-entropy loss = -Σlog P(y_i|x_i). Minimizing cross-entropy = maximizing log-likelihood = MLE. So every time you train a classifier with cross-entropy loss, you're doing MLE.

**Q: What is the KL divergence and why can't it be used as a distance metric?**
A: KL(P||Q) = Σ P(x) log(P(x)/Q(x)) measures "extra bits needed to encode P using Q's code." It's not symmetric: KL(P||Q) ≠ KL(Q||P). Not a metric. Forward KL (KL(data||model)) = mode-covering; reverse KL (KL(model||data)) = mode-seeking. VAEs minimize reverse KL.

**Q: Explain the connection between L2 regularization and Gaussian priors.**
A: MAP estimation: maximize P(θ|data) ∝ P(data|θ)·P(θ). If P(θ) = N(0, σ²I), then log P(θ) = -||θ||²/(2σ²) + const. Adding this to log-likelihood gives: maximize log P(data|θ) - λ||θ||². This is exactly L2 regularization with λ = 1/(2σ²). L2 reg = assuming Gaussian prior on weights.

**Q: Why does Adam work better than SGD for most deep learning tasks?**
A: Adam maintains per-parameter adaptive learning rates using first moment (mean of gradients, like momentum) and second moment (mean of squared gradients, like RMSProp). Parameters with large gradients get smaller updates; sparse/small-gradient parameters get larger updates. This helps with protein embeddings where different residue positions have very different gradient magnitudes.

### Mathematical Operations — Time Complexity
| Operation | Time | Space | Notes |
|-----------|------|-------|-------|
| Matrix multiply (n×n) | O(n³) naive, O(n^2.37) Strassen | O(n²) | GPU uses BLAS |
| Eigendecomposition | O(n³) | O(n²) | Full decomposition |
| SVD (n×m, n>m) | O(n·m²) | O(n·m) | Truncated SVD: O(n·m·k) |
| PCA (top-k) | O(n·d²+d³) | O(d²) | Or randomized SVD O(n·d·k) |
| KL divergence | O(n) | O(1) | n = number of states |
| Backprop | O(edges in graph) | O(activations) | ~2-3× forward |

In [ ]:
# Mathematical Foundations: Summary
import numpy as np
from scipy import stats

np.random.seed(42)

# 1. Linear algebra: matrix decomposition
A = np.random.randn(4, 4)
U, s, Vt = np.linalg.svd(A)
print(f"Condition number of A: {s[0]/s[-1]:.2f}")
print(f"Rank of A: {np.linalg.matrix_rank(A)}")

# 2. Probability: sample from distribution
samples = np.random.normal(0, 1, 1000)
print(f"\nSample stats: mean={samples.mean():.3f}, std={samples.std():.3f}")
print(f"KS test vs N(0,1): p={stats.kstest(samples, 'norm').pvalue:.3f}")

# 3. Optimization: gradient step
def rosenbrock(x): return (1-x[0])**2 + 100*(x[1]-x[0]**2)**2
def rosenbrock_grad(x):
    return np.array([
        -2*(1-x[0]) - 400*x[0]*(x[1]-x[0]**2),
        200*(x[1]-x[0]**2)
    ])

x = np.array([-1.0, 0.5])
for _ in range(1000):
    x -= 1e-3 * rosenbrock_grad(x)
print(f"\nRosenbrock minimum found: {x} (true: [1,1])")

## Mastery Check

Before moving on, make sure you can:
1. explain the core concept of this notebook in plain English without looking
2. write a small toy example from scratch
3. identify one common mistake and explain why it is wrong
4. say whether you should revisit math, Python, or ML basics before continuing


## 📦 Real Datasets for Mathematical Foundations

The following datasets are used to ground the linear algebra and calculus concepts in this notebook with real-world examples.

### Dataset 1: MNIST Handwritten Digits (via torchvision)
- **URL:** Downloaded automatically by torchvision from http://yann.lecun.com/exdb/mnist/
- **Format:** 28×28 grayscale images, 60,000 train / 10,000 test
- **Size:** ~11 MB compressed
- **Why it matters:** MNIST is the 'hello world' of ML. Each image is a 784-dimensional vector — perfect for visualizing PCA, dot products, and matrix decompositions in a tangible way.

### Dataset 2: NumPy built-in random datasets
- **Source:** `numpy.random` — no download needed
- **Why it matters:** Generating structured random data lets you test linear algebra operations (eigendecomposition, SVD) with controlled inputs.

_Note: If the MNIST download fails (no internet), the simulated data in this notebook still teaches the same linear algebra concepts._


In [ ]:
# ── Real Dataset: MNIST via torchvision ─────────────────────────────────────
# Install if needed: pip install torchvision
try:
    import torchvision
    import torchvision.transforms as transforms
    import numpy as np
    import matplotlib.pyplot as plt

    # Download MNIST (will cache to ./data/ after first download)
    mnist_train = torchvision.datasets.MNIST(
        root='./data', train=True, download=True,
        transform=transforms.ToTensor()
    )
    mnist_test = torchvision.datasets.MNIST(
        root='./data', train=False, download=True,
        transform=transforms.ToTensor()
    )

    # Convert to numpy for linear algebra exercises
    # Shape: (60000, 784) — each row is one flattened 28×28 image
    X_train = mnist_train.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
    y_train = mnist_train.targets.numpy()

    print(f'MNIST training set shape: {X_train.shape}')
    print(f'Labels (0-9): {np.unique(y_train)}')

    # Visualize 5 examples
    fig, axes = plt.subplots(1, 5, figsize=(10, 2))
    for i, ax in enumerate(axes):
        ax.imshow(X_train[i].reshape(28, 28), cmap='gray')
        ax.set_title(f'Label: {y_train[i]}')
        ax.axis('off')
    plt.suptitle('MNIST samples — each is a 784-dim vector')
    plt.tight_layout()
    plt.show()

    # PCA on a subset — pure numpy linear algebra
    X_subset = X_train[:2000]  # use 2000 samples for speed
    X_centered = X_subset - X_subset.mean(axis=0)
    # Covariance matrix (784 x 784)
    cov = np.cov(X_centered.T)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    # Sort descending
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    explained = eigenvalues / eigenvalues.sum()
    print(f'Top 10 eigenvalues explain {explained[:10].sum()*100:.1f}% of variance')

except ImportError:
    print('torchvision not installed. Run: pip install torchvision')
    print('Falling back to simulated data...')
    import numpy as np
    # Simulated 784-dim data (same shape as MNIST)
    np.random.seed(42)
    X_train = np.random.randn(1000, 784).astype(np.float32)
    y_train = np.random.randint(0, 10, 1000)
    print(f'Simulated data shape: {X_train.shape}')


## Notebook Complete

**You can now:**
- Apply linear algebra (matrix ops, SVD, eigendecomposition) relevant to ML
- Derive and implement gradient descent and backpropagation by hand

**Where this fits:**
- Previous: 07_tensorflow_keras
- Next: 09_classical_ml_advanced — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes